In [48]:
import pandas as pd

# Test reading and calling a specific row from the master photometry list. 
df = pd.read_csv('/Users/cwray/Desktop/ASTR502/ASTR502-SP26/ASTR502-SP26/ASTR502_Master_Photometry_List.csv', sep=' ,')
# Iloc calls specified rows by index number.
df = df.iloc[[4256]]
print(df)

     pl_name,hostname,gaia_dr3_id,ra,dec,fuvmag,e_fuvmag,nuvmag,e_nuvmag,umag,e_umag,gmag,e_gmag,rmag,e_rmag,imag,e_imag,zmag,e_zmag,vmag,e_vmag,bmag,e_bmag,gprimemag,e_gprimemag,rprimemag,e_rprimemag,iprimemag,e_iprimemag,gaiaGmag,e_gaiaGmag,gaiaBPmag,e_gaiaBPmag,gaiaRPmag,e_gaiaRPmag,gmag,e_gmag,rmag,e_rmag,imag,e_imag,zmag,e_zmag,ymag,e_ymag,Jmag,e_Jmag,Hmag,e_Hmag,Kmag,e_Kmag,w1mag,e_w1mag,w2mag,e_w2mag,w3mag,e_w3mag,w4mag,e_w4mag
4256  HD 209458 b,HD 209458,1779546757669063552,330....                                                                                                                                                                                                                                                                                                                                                                                               


C:\Users\cwray\AppData\Local\Temp\ipykernel_12232\2366023596.py:4: ParserWarning: Falling back to the 'python' engine because the 'c' engine does not support regex separators (separators > 1 char and different from '\s+' are interpreted as regex); you can avoid this warning by specifying engine='python'.
  df = pd.read_csv('/Users/cwray/Desktop/ASTR502/ASTR502-SP26/ASTR502-SP26/ASTR502_Master_Photometry_List.csv', sep=' ,')


In [49]:
import pandas as pd
import numpy as np
from scipy.interpolate import NearestNDInterpolator

# 1. Load fresh
master_df = pd.read_csv('/Users/cwray/Desktop/ASTR502/ASTR502-SP26/ASTR502-SP26/ASTR502_Mega_Target_List.csv')

# 2. Extract exactly what we need into a clean NumPy array
# We use .copy() to ensure we aren't working on a "view" of the data
grid_data = master_df[['st_age', 'st_met']].copy()

# 3. Drop any rows that have 'NaN' (Missing values) 
# Optimizers will crash if there are holes in your master list
grid_data = grid_data.dropna()

# 4. Define your grid points
grid_points = grid_data.values 

print(f"Grid shape: {grid_points.shape}") # Should show (Number of stars, 2)

Grid shape: (3762, 2)


In [50]:
# Create the interpolator using the same cleaned indices
interp_teff = NearestNDInterpolator(grid_points, master_df.loc[grid_data.index, 'st_teff'])
interp_vmag = NearestNDInterpolator(grid_points, master_df.loc[grid_data.index, 'sy_vmag'])

In [51]:
def objective_function(guess, observed_teff, observed_met, mode="spectro"):
    age_guess, met_guess = guess
    
    # 1. Ask the interpolator what the Teff would be for this guess
    predicted_teff = interp_teff(age_guess, met_guess)
    predicted_vmag = interp_vmag(age_guess, met_guess)
    
    if mode == "spectro":
        # Compare Teff: (Observed - Predicted)^2
        return (observed_teff - predicted_teff)**2
    else:
        # This is the "Raw Photometry" test mentioned in your prompt
        # We use a dummy observed Vmag to see if it fails
        observed_vmag = 4.83 # Example for Sun
        return (observed_vmag - predicted_vmag)**2

In [52]:
from scipy.optimize import minimize

# Define your known test star (e.g., HD 209458)
target_teff = 6092 
target_met = 0.02

# Initial guess: [Age=5.0, Metallicity=0.0]
initial_guess = [5.0, 0.0]

# RUN THE TEST
result = minimize(objective_function, initial_guess, 
                  args=(target_teff, target_met, "spectro"),
                  bounds=[(0.1, 13.0), (-1.0, 0.5)])

print(f"Calculated Age: {result.x[0]:.2f} Gyr")

Calculated Age: 5.00 Gyr


In [53]:
# New test
import numpy as np
import pandas as pd
from scipy.interpolate import LinearNDInterpolator
from scipy.optimize import minimize

# --- LOAD DATA ---
df = pd.read_csv('/Users/cwray/Desktop/ASTR502/ASTR502-SP26/ASTR502-SP26/ASTR502_Mega_Target_List.csv')
df.columns = df.columns.str.strip()

# Drop rows with missing critical data to ensure the grid is clean
clean_df = df.dropna(subset=['st_age', 'st_met', 'st_mass', 'st_teff', 'abs_vmag']).copy()

# 1. Define the Grid (The "Model" coordinates)
# We use Age, Metallicity, and Mass as our search dimensions
grid_points = clean_df[['st_age', 'st_met', 'st_mass']].values

# 2. Define the Interpolators (The "Predictors")
# These allow the code to guess what Teff or Vmag should be for any age/met/mass
interp_teff = LinearNDInterpolator(grid_points, clean_df['st_teff'])
interp_vmag = LinearNDInterpolator(grid_points, clean_df['abs_vmag'])

In [54]:
def objective_function(params, target_val, mode="spectro"):
    age_guess, met_guess, mass_guess = params
    
    # Get the model's prediction for this guess
    pred_teff = interp_teff(age_guess, met_guess, mass_guess)
    pred_vmag = interp_vmag(age_guess, met_guess, mass_guess)
    
    if mode == "spectro":
        # Comparing Predicted Teff to Observed Teff
        return (target_val - pred_teff)**2
    else:
        # Comparing Predicted Abs Vmag to Observed Abs Vmag
        # This is the "Raw Photometry" test
        return (target_val - pred_vmag)**2

In [55]:
# Known Sun Values
sun_teff = 6091.0
sun_vmag = 4.24044588815839
sun_true_age = 3.1

# Initial Guess [Age, Met, Mass]
x0 = [5.0, 0.0, 1.0]
bounds = [(0.01, 13.8), (-0.5, 0.5), (0.1, 2.0)]

# --- TEST 1: SPECTROSCOPIC (Teff) ---
res_spec = minimize(
    objective_function, 
    x0, 
    args=(sun_teff, "spectro"), 
    bounds=bounds,
    method='L-BFGS-B',
    options={'eps': 0.1} # This is the "magic" fix for flat grids
)

# --- TEST 2: PHOTOMETRIC (Abs Vmag) ---
res_phot = minimize(objective_function, x0, args=(sun_vmag, "photo"), bounds=bounds)

print(f"--- RESULTS FOR THE SUN ---")
print(f"True Age: {sun_true_age} Gyr")
print(f"Spectro Age Output: {res_spec.x[0]:.3f} Gyr")
print(f"Photo Age Output:   {res_phot.x[0]:.3f} Gyr")

--- RESULTS FOR THE SUN ---
True Age: 3.1 Gyr
Spectro Age Output: 5.000 Gyr
Photo Age Output:   5.000 Gyr


In [56]:
print("Manual check of the landscape:")
for test_age in [3.0, 4.0, 5.0, 6.0]:
    val = interp_teff(test_age, 0.0, 1.0) # Age, Met, Mass
    print(f"Age: {test_age} -> Predicted Teff: {val}")

Manual check of the landscape:
Age: 3.0 -> Predicted Teff: 5724.304935099163
Age: 4.0 -> Predicted Teff: 5800.738955823293
Age: 5.0 -> Predicted Teff: 5725.204526404023
Age: 6.0 -> Predicted Teff: 5823.751707672923
